In [ ]:
import pandas as pd

df = pd.read_csv("../data/raw/helpdesk_customer_tickets.csv")   # change the filename

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nMissing values:\n", df.isna().sum())

# Show one row in full
print("\nSample row:\n", df.iloc[0])

# Show value counts for every short-text column (the label-like ones)
for col in df.columns:
    if df[col].dtype == "object" and df[col].nunique() < 30:
        print(f"\n--- {col} ---")
        print(df[col].value_counts(dropna=False))

In [ ]:
print("Language:\n", df["language"].value_counts(), "\n")

en = df[df["language"] == "en"].copy()
print("English rows:", len(en), "\n")

print("Queue (English):\n", en["queue"].value_counts(), "\n")
print("Priority (English):\n", en["priority"].value_counts(), "\n")
print("Type (English):\n", en["type"].value_counts(), "\n")

# Tags carry useful extra labels
tags = pd.concat([en[f"tag_{i}"] for i in range(1, 10)]).dropna()
print("Top tags (English):\n", tags.value_counts().head(25), "\n")

print("Duplicate bodies:", en["body"].duplicated().sum())
print("Missing subject:", en["subject"].isna().sum())

In [ ]:
for q in ["Customer Service", "Product Support"]:
    print("=====", q)
    for _, r in en[en["queue"] == q].sample(6, random_state=2).iterrows():
        print("-", r["subject"], "|", str(r["body"])[:250], "\n")

In [ ]:
import re, random
random.seed(42)

en = df[df["language"] == "en"].copy()

QUEUE_MAP = {
    "Technical Support": "Technical Support",
    "IT Support": "Technical Support",
    "Service Outages and Maintenance": "Technical Support",
    "Product Support": "Technical Support",
    "Customer Service": "Technical Support",
    "Billing and Payments": "Payment Issue",
    "Returns and Exchanges": "Refund Request",
}
en = en[en["queue"].isin(QUEUE_MAP)].copy()
en["category"] = en["queue"].map(QUEUE_MAP)

# Keyword overrides for the two noisy queues
sales_kw = r"availability|in stock|delivery|pricing|price|quote|purchase|order"
inquiry_kw = r"compatib|inquiry|recommend|information about|upgrade"

text = (en["subject"].fillna("") + " " + en["body"]).str.lower()
noisy = en["queue"].isin(["Product Support", "Customer Service"])

en["needs_review"] = False
m_sales = noisy & text.str.contains(sales_kw)
m_inq = noisy & ~m_sales & text.str.contains(inquiry_kw)
en.loc[m_sales, ["category", "needs_review"]] = ["Sales Inquiry", True]
en.loc[m_inq, ["category", "needs_review"]] = ["General Inquiry", True]
# Billing/returns rows also get a quick look
en.loc[en["queue"].isin(["Billing and Payments", "Returns and Exchanges"]), "needs_review"] = True

# Replace placeholders with realistic values
names = ["Alex", "Priya", "John", "Sara", "Ravi", "Maria", "David", "Anita"]
def fill(t):
    t = re.sub(r"<name>", lambda m: random.choice(names), str(t))
    t = re.sub(r"<acc_num>", lambda m: str(random.randint(100000, 999999)), t)
    return t

out = pd.DataFrame({
    "email_id": ["K" + str(i).zfill(4) for i in range(len(en))],
    "subject": en["subject"].fillna("No subject").map(fill).values,
    "body": en["body"].map(fill).values,
    "category": en["category"].values,
    "priority": en["priority"].str.capitalize().values,
    "source": "kaggle",
    "needs_review": en["needs_review"].values,
})

print(out["category"].value_counts())
print("\nRows to review:", out["needs_review"].sum())
out.to_csv("../data/raw/kaggle_en_mapped.csv", index=False)

In [ ]:
pd.set_option("display.max_colwidth", 160)
print(out[out["needs_review"]][["email_id", "subject", "category"]].to_string())

In [ ]:
check = ["K0000", "K0024", "K0080", "K0117", "K0126", "K0139", "K0143"]
for _, r in out[out["email_id"].isin(check)].iterrows():
    print(r["email_id"], "|", r["category"], "|", r["subject"])
    print(r["body"][:350], "\n")

In [ ]:
fixes = {
    "K0013": "Technical Support", "K0062": "Technical Support",
    "K0090": "Technical Support", "K0093": "Technical Support",
    "K0134": "Technical Support", "K0135": "Technical Support",
    "K0041": "General Inquiry",   "K0095": "Refund Request",
    # add your decisions from the body check, for example:
    # "K0117": "Technical Support", "K0126": "Technical Support",
    # "K0139": "General Inquiry",   "K0143": "Technical Support",
}
for eid, cat in fixes.items():
    out.loc[out["email_id"] == eid, "category"] = cat

out["needs_review"] = False
print(out["category"].value_counts())
out.to_csv("../data/raw/kaggle_en_clean.csv", index=False)

In [ ]:
print(out.loc[out["email_id"] == "K0024", "body"].values[0])

In [ ]:
fixes2 = {
    "K0000": "Technical Support",
    "K0024": "Technical Support",   # change to Refund Request if the body asks for a return
    "K0080": "General Inquiry",
    "K0117": "Technical Support",
    "K0126": "Refund Request",
    "K0139": "General Inquiry",
    "K0143": "Technical Support",
}
for eid, cat in fixes2.items():
    out.loc[out["email_id"] == eid, "category"] = cat

print(out["category"].value_counts())
out.to_csv("../data/raw/kaggle_en_clean.csv", index=False)

In [ ]:
import pandas as pd

d = pd.read_csv("../data/synthetic/payment_1.csv")

print("Rows:", len(d))
print("Columns:", d.columns.tolist())
print("\nPriority:\n", d["priority"].value_counts())

# Length variety (words per email)
d["words"] = d["body"].str.split().str.len()
print("\nLength:\n", d["words"].describe())

# Repeated openings
print("\nMost common openings:\n", d["body"].str[:20].value_counts().head(8))

# Duplicates
print("\nDuplicate bodies:", d["body"].duplicated().sum())

# Read a random sample
pd.set_option("display.max_colwidth", 300)
print(d.sample(6, random_state=1)[["subject", "body", "priority"]].to_string())

In [ ]:
kw = ["twice|two times|2x|double", "pending|no order|not confirmed", "declin|3d secure|failed",
      "wrong amount|overcharg|incorrect", "invoice|gst|receipt", "cancel", "emi|offer|coupon|discount"]
for k in kw:
    print(k, "->", d["body"].str.lower().str.contains(k).sum())

In [ ]:
import pandas as pd

d = pd.read_csv("../data/synthetic/refund_1.csv")

print("Rows:", len(d))
print("Columns:", d.columns.tolist())
print("\nPriority:\n", d["priority"].value_counts())

# Length variety (words per email)
d["words"] = d["body"].str.split().str.len()
print("\nLength:\n", d["words"].describe())

# Repeated openings
print("\nMost common openings:\n", d["body"].str[:20].value_counts().head(8))

# Duplicates
print("\nDuplicate bodies:", d["body"].duplicated().sum())

# Read a random sample
pd.set_option("display.max_colwidth", 300)
print(d.sample(6, random_state=1)[["subject", "body", "priority"]].to_string())

In [ ]:
kw = ["defect|faulty|damaged|flicker|broken", "cancel", "wrong item|incorrect item",
      "not received|nothing|still waiting|overdue", "store credit|voucher", "partial|part of"]
for k in kw:
    print(k, "->", d["body"].str.lower().str.contains(k).sum())

In [ ]:
d = pd.read_csv("../data/synthetic/account_1.csv")

print("Rows:", len(d))
print("Columns:", d.columns.tolist())
print("\nPriority:\n", d["priority"].value_counts())

# Length variety (words per email)
d["words"] = d["body"].str.split().str.len()
print("\nLength:\n", d["words"].describe())

# Repeated openings
print("\nMost common openings:\n", d["body"].str[:20].value_counts().head(8))

# Duplicates
print("\nDuplicate bodies:", d["body"].duplicated().sum())

# Read a random sample
pd.set_option("display.max_colwidth", 300)
print(d.sample(6, random_state=1)[["subject", "body", "priority"]].to_string())

In [ ]:
print(d.isna().sum())
print(d.iloc[29])

In [ ]:
d = pd.read_csv("../data/synthetic/sales_1.csv")

print("Rows:", len(d))
print("Columns:", d.columns.tolist())
print("\nPriority:\n", d["priority"].value_counts())

# Length variety (words per email)
d["words"] = d["body"].str.split().str.len()
print("\nLength:\n", d["words"].describe())

# Repeated openings
print("\nMost common openings:\n", d["body"].str[:20].value_counts().head(8))

# Duplicates
print("\nDuplicate bodies:", d["body"].duplicated().sum())

# Read a random sample
pd.set_option("display.max_colwidth", 300)
print(d.sample(6, random_state=1)[["subject", "body", "priority"]].to_string())

In [ ]:
d = pd.read_csv("../data/synthetic/complaint_1.csv")

print("Rows:", len(d))
print("Columns:", d.columns.tolist())
print("\nPriority:\n", d["priority"].value_counts())

# Length variety (words per email)
d["words"] = d["body"].str.split().str.len()
print("\nLength:\n", d["words"].describe())

# Repeated openings
print("\nMost common openings:\n", d["body"].str[:20].value_counts().head(8))

# Duplicates
print("\nDuplicate bodies:", d["body"].duplicated().sum())

# Read a random sample
pd.set_option("display.max_colwidth", 300)
print(d.sample(6, random_state=1)[["subject", "body", "priority"]].to_string())

In [ ]:
d = pd.read_csv("../data/synthetic/job_1.csv")

print("Rows:", len(d))
print("Columns:", d.columns.tolist())
print("\nPriority:\n", d["priority"].value_counts())

# Length variety (words per email)
d["words"] = d["body"].str.split().str.len()
print("\nLength:\n", d["words"].describe())

# Repeated openings
print("\nMost common openings:\n", d["body"].str[:20].value_counts().head(8))

# Duplicates
print("\nDuplicate bodies:", d["body"].duplicated().sum())

# Read a random sample
pd.set_option("display.max_colwidth", 300)
print(d.sample(6, random_state=1)[["subject", "body", "priority"]].to_string())

In [ ]:
d = pd.read_csv("../data/synthetic/general_1.csv")

print("Rows:", len(d))
print("Columns:", d.columns.tolist())
print("\nPriority:\n", d["priority"].value_counts())

# Length variety (words per email)
d["words"] = d["body"].str.split().str.len()
print("\nLength:\n", d["words"].describe())

# Repeated openings
print("\nMost common openings:\n", d["body"].str[:20].value_counts().head(8))

# Duplicates
print("\nDuplicate bodies:", d["body"].duplicated().sum())

# Read a random sample
pd.set_option("display.max_colwidth", 300)
print(d.sample(6, random_state=1)[["subject", "body", "priority"]].to_string())

In [ ]:
import pandas as pd, glob, os

KEY = {"payment": "Payment Issue", "refund": "Refund Request",
       "account": "Account & Login", "sales": "Sales Inquiry",
       "complaint": "Complaint", "job": "Job Application",
       "general": "General Inquiry", "tech": "Technical Support"}

parts = []
for f in sorted(glob.glob("../data/synthetic/*.csv")):
    name = os.path.basename(f)
    key = name.split("_")[0]
    if key not in KEY:
        print("SKIPPED (unknown key):", name)   # catches typos like payment1.csv or tech-1.csv
        continue
    d = pd.read_csv(f)
    d["category"] = KEY[key]
    d["source"] = "synthetic"
    d["batch"] = name
    parts.append(d)

syn = pd.concat(parts, ignore_index=True)
syn["subject"] = syn["subject"].fillna("No subject")
syn["priority"] = syn["priority"].str.strip().str.capitalize()
syn = syn.drop_duplicates(subset="body")
syn = syn[syn["body"].str.len() > 15]

real = pd.read_csv("../data/raw/kaggle_en_clean.csv").drop(columns=["needs_review"], errors="ignore")
real["batch"] = "kaggle"
full = pd.concat([real, syn], ignore_index=True)
full["email_id"] = ["E" + str(i).zfill(4) for i in range(len(full))]

print(full["category"].value_counts())
print(full.groupby(["category", "source"]).size().unstack(fill_value=0))
print("\nFiles merged:", len(parts), "| rows per file:")
print(full[full["source"] == "synthetic"].groupby("batch").size().to_string())
full.to_csv("../data/emails_full.csv", index=False)

In [ ]:
import pandas as pd
full = pd.read_csv("../data/emails_full.csv")
full["words"] = full["body"].str.split().str.len()

# 1. Priority by category (this feeds the priority model later)
print(pd.crosstab(full["category"], full["priority"]))

# 2. Length by category (big gaps let a model cheat on length)
print(full.groupby("category")["words"].agg(["mean", "median"]).round(0))

# 3. Repeated openings across the whole dataset
print(full["body"].str[:30].value_counts().head(10))

# 4. Same body text appearing in more than one category
dup = full[full.duplicated("body", keep=False)]
print("Cross-category duplicates:", len(dup))

# 5. Product-name leakage from the Kaggle rows
leak = full["body"].str.contains("Dell|Canon|Zoom|Jira|MacBook|PIXMA", case=False)
print(pd.crosstab(full["category"], leak))

In [ ]:
tech = full[full["category"] == "Technical Support"]
keep = pd.concat([
    tech[tech["source"] == "synthetic"],
    tech[tech["source"] == "kaggle"].sample(20, random_state=42),
])
final = pd.concat([full[full["category"] != "Technical Support"], keep], ignore_index=True)
final = final.sample(frac=1, random_state=42).reset_index(drop=True)
print(final["category"].value_counts())
final.to_csv("../data/emails_final.csv", index=False)

In [ ]:
from sklearn.model_selection import train_test_split

train, temp = train_test_split(final, test_size=0.30, stratify=final["category"], random_state=42)
val, test = train_test_split(temp, test_size=0.50, stratify=temp["category"], random_state=42)

for name, d in [("train", train), ("val", val), ("test", test)]:
    d.to_csv(f"../data/{name}.csv", index=False)
    print(name, len(d))
print(test["category"].value_counts())

In [ ]:
import re
k = final["source"] == "kaggle"
for col in ["subject", "body"]:
    final.loc[k, col] = final.loc[k, col].str.replace(r"Tech Online Store\s*", "", flags=re.I, regex=True)
final.to_csv("../data/emails_final.csv", index=False)

In [ ]:
import sys, time
sys.path.append("../src")
import pandas as pd
from preprocess import preprocess_df
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, f1_score

train = pd.read_csv("../data/train.csv")
val = pd.read_csv("../data/val.csv")
X_tr, X_va = preprocess_df(train), preprocess_df(val)
y_tr, y_va = train["category"], val["category"]

models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=2000, C=10),
    "Linear SVM (calibrated)": CalibratedClassifierCV(LinearSVC(), cv=5),
}
rows = []
for name, clf in models.items():
    pipe = Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
                     ("clf", clf)])
    t0 = time.time(); pipe.fit(X_tr, y_tr); fit_s = time.time() - t0
    t0 = time.time(); pred = pipe.predict(X_va); ms = (time.time() - t0) / len(X_va) * 1000
    rows.append({"model": name,
                 "accuracy": accuracy_score(y_va, pred),
                 "macro_f1": f1_score(y_va, pred, average="macro"),
                 "weighted_f1": f1_score(y_va, pred, average="weighted"),
                 "train_s": fit_s, "ms_per_email": ms})
print(pd.DataFrame(rows).round(3).to_string(index=False))

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline

trval = pd.concat([train, val], ignore_index=True)
X, y = preprocess_df(trval), trval["category"]
cv = StratifiedKFold(5, shuffle=True, random_state=42)

for name, clf in models.items():
    pipe = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True), clf)
    s = cross_validate(pipe, X, y, cv=cv, scoring=["accuracy", "f1_macro"])
    print(f"{name:26s} acc {s['test_accuracy'].mean():.3f} ± {s['test_accuracy'].std():.3f} | "
          f"macro F1 {s['test_f1_macro'].mean():.3f} ± {s['test_f1_macro'].std():.3f}")

In [ ]:
import os, matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

os.makedirs("../reports", exist_ok=True)
best = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
                     LogisticRegression(max_iter=2000, C=10))
best.fit(X_tr, y_tr)
pred = best.predict(X_va)

print(classification_report(y_va, pred, digits=3))

labels = sorted(y_va.unique())
cm = confusion_matrix(y_va, pred, labels=labels)
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(cm, display_labels=labels).plot(ax=ax, xticks_rotation=45, cmap="Blues", colorbar=False)
plt.tight_layout(); plt.savefig("../reports/cm_tfidf_lr.png", dpi=150); plt.show()

mask = pred != y_va.values
err = val[mask].copy(); err["predicted"] = pred[mask]
pd.set_option("display.max_colwidth", 220)
print(err[["category", "predicted", "subject", "body"]].to_string())

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np, time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

st = SentenceTransformer("all-MiniLM-L6-v2")
T_tr = preprocess_df(train, mode="embedding")
T_va = preprocess_df(val, mode="embedding")

E_tr = st.encode(T_tr, batch_size=32, normalize_embeddings=True, show_progress_bar=True)
t0 = time.time()
E_va = st.encode(T_va, batch_size=32, normalize_embeddings=True)
embed_ms = (time.time() - t0) / len(T_va) * 1000
np.save("../data/E_tr.npy", E_tr); np.save("../data/E_va.npy", E_va)

# A) Logistic Regression on embeddings
lr = LogisticRegression(max_iter=3000, C=10).fit(E_tr, y_tr)
t0 = time.time(); pred_lr = lr.predict(E_va); pred_ms = (time.time() - t0) / len(E_va) * 1000

# B) Similarity only: nearest category centroid (cosine)
cats = sorted(y_tr.unique())
cent = np.vstack([E_tr[(y_tr == c).values].mean(0) for c in cats])
cent /= np.linalg.norm(cent, axis=1, keepdims=True)
pred_cent = np.array(cats)[(E_va @ cent.T).argmax(1)]

for name, p in [("MiniLM + Logistic Regression", pred_lr), ("MiniLM + nearest centroid", pred_cent)]:
    print(f"{name:30s} acc {accuracy_score(y_va, p):.3f} | macro F1 {f1_score(y_va, p, average='macro'):.3f}")
print(f"embedding time {embed_ms:.1f} ms/email + classify {pred_ms:.2f} ms/email")

# Which emails does it get wrong?
bad = pred_lr != y_va.values
e2 = val[bad].copy(); e2["predicted"] = pred_lr[bad]
print(e2[["category", "predicted", "subject"]].to_string())

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline

trval = pd.concat([train, val], ignore_index=True)
Xt = np.array(preprocess_df(trval), dtype=object)
E_all = np.vstack([E_tr, E_va])            # same row order as trval
y = trval["category"].values
cv = StratifiedKFold(5, shuffle=True, random_state=42)

def tfidf_lr():
    return make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
                         LogisticRegression(max_iter=2000, C=10))

res = {"TF-IDF + LR": [], "MiniLM + LR": [], "Ensemble (avg probs)": []}
for tr_i, te_i in cv.split(Xt, y):
    a = tfidf_lr().fit(Xt[tr_i], y[tr_i])
    b = LogisticRegression(max_iter=3000, C=10).fit(E_all[tr_i], y[tr_i])
    pa, pb = a.predict_proba(Xt[te_i]), b.predict_proba(E_all[te_i])
    cls = a.classes_                      # same order as b.classes_
    for k, p in [("TF-IDF + LR", pa), ("MiniLM + LR", pb), ("Ensemble (avg probs)", (pa + pb) / 2)]:
        pred = cls[p.argmax(1)]
        res[k].append((accuracy_score(y[te_i], pred), f1_score(y[te_i], pred, average="macro")))

for k, v in res.items():
    v = np.array(v)
    print(f"{k:22s} acc {v[:,0].mean():.3f} ± {v[:,0].std():.3f} | macro F1 {v[:,1].mean():.3f} ± {v[:,1].std():.3f}")

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"

path = DATA / "test_handwritten.csv"
print("looking for:", path, "| exists:", path.exists())

In [ ]:
if not path.exists():
    pd.DataFrame(columns=["id","subject","body","category","priority",
                          "type","alt_label","split"]).to_csv(path, index=False)

In [ ]:
import numpy as np

cats8 = ["Technical Support","Payment Issue","Account & Login","Sales Inquiry",
         "Complaint","Refund Request","Job Application","General Inquiry"]
plan = {"short":3, "messy":3, "borderline":2, "novel":2}
unk_groups = ["spam","newsletter","wrongrecip","autoreply","nonenglish","unrelated"]

rows = []
for c in cats8:
    slug = c.split()[0].lower()[:4]
    for t, n in plan.items():
        for i in range(1, n+1):
            rows.append(dict(id=f"{slug}-{t}-{i}", subject="", body="", category=c,
                             priority="", type=t, alt_label="", split="test"))

rng = np.random.default_rng(42)
for g in unk_groups:
    dev_idx = set(rng.choice(5, size=2, replace=False) + 1)   # 2 dev per group
    for i in range(1, 6):
        rows.append(dict(id=f"unk-{g}-{i}", subject="", body="", category="Unknown",
                         priority="NA", type="unknown", alt_label="",
                         split="dev" if i in dev_idx else "test"))

scaffold = pd.DataFrame(rows)
out = DATA / "test_handwritten.csv"
if out.exists() and len(pd.read_csv(out)) > 0:
    print("File already has rows, not overwriting.")
else:
    scaffold.to_csv(out, index=False, encoding="utf-8")
    print(len(scaffold), "rows (80 category + 30 Unknown); dev:", (scaffold.split=="dev").sum())

In [ ]:
import sys, importlib
sys.path.insert(0, str(ROOT))          # lets "from src..." work from notebooks/

from src import decision
importlib.reload(decision)             # picks up edits to decision.py without restarting the kernel
from src.decision import load_config, decide

cfg = load_config(ROOT / "config" / "pipeline.yaml")

In [ ]:
from sklearn.base import BaseEstimator
for name, obj in list(globals().items()):
    if isinstance(obj, BaseEstimator) and hasattr(obj, "classes_"):
        print(name, type(obj).__name__)
    elif callable(obj) and any(k in name.lower() for k in ["tfidf", "minilm", "ensemble", "proba"]):
        print(name, "-> function")

In [ ]:
from sklearn.base import BaseEstimator
found = [(n, type(o).__name__) for n, o in globals().items()
         if isinstance(o, BaseEstimator) and hasattr(o, "classes_")]
print(found if found else "no fitted classifiers in this kernel")

In [ ]:
from src.preprocess import preprocess_df
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sentence_transformers import SentenceTransformer

X_tr_t = preprocess_df(train, mode="tfidf")
X_va_t = preprocess_df(val,   mode="tfidf")
X_tr_e = preprocess_df(train, mode="embedding")
X_va_e = preprocess_df(val,   mode="embedding")

vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)   # swap in your CV settings
lr_t = LogisticRegression(max_iter=2000).fit(vec.fit_transform(X_tr_t), train.category)

st = SentenceTransformer("all-MiniLM-L6-v2")
lr_e = LogisticRegression(max_iter=2000).fit(st.encode(list(X_tr_e)), train.category)

assert (lr_t.classes_ == lr_e.classes_).all()
classes = np.array(lr_t.classes_)
assert set(classes) == set(cfg["categories"]), set(classes) ^ set(cfg["categories"])

proba_val = (lr_t.predict_proba(vec.transform(X_va_t)) +
             lr_e.predict_proba(st.encode(list(X_va_e)))) / 2
print(proba_val.shape, classes)

In [ ]:
res = pd.DataFrame([decide(p, classes, cfg) for p in proba_val])
res["correct"] = res.predicted.values == val.category.values
print(res.groupby("action").agg(n=("correct","size"), acc=("correct","mean")))
print(res.groupby("correct").confidence.describe())

In [ ]:
from scipy.optimize import minimize_scalar
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline

skf = StratifiedKFold(5, shuffle=True, random_state=42)
y_tr = train.category.values
E_tr = st.encode(list(X_tr_e))
E_va = st.encode(list(X_va_e))

def oof(model, X):
    return cross_val_predict(model, X, y_tr, cv=skf, method="predict_proba")

# 1) pick C for each model by out-of-fold log loss (train only)
best = {}
for name, mk, X in [
    ("tfidf", lambda C: make_pipeline(TfidfVectorizer(ngram_range=(1,2), sublinear_tf=True),
                                      LogisticRegression(C=C, max_iter=3000)), X_tr_t),
    ("emb",   lambda C: LogisticRegression(C=C, max_iter=3000), E_tr),
]:
    scores = {C: log_loss(y_tr, oof(mk(C), X), labels=classes) for C in [1, 10, 30, 100, 300]}
    best[name] = min(scores, key=scores.get)
    print(name, {k: round(v, 3) for k, v in scores.items()}, "-> C =", best[name])

# 2) OOF ensemble, then fit one temperature on it
P_oof = (oof(make_pipeline(TfidfVectorizer(ngram_range=(1,2), sublinear_tf=True),
                           LogisticRegression(C=best["tfidf"], max_iter=3000)), X_tr_t) +
         oof(LogisticRegression(C=best["emb"], max_iter=3000), E_tr)) / 2

def sharpen(P, T):
    Q = np.power(P, 1.0 / T)
    return Q / Q.sum(axis=1, keepdims=True)

T = minimize_scalar(lambda t: log_loss(y_tr, sharpen(P_oof, t), labels=classes),
                    bounds=(0.1, 3.0), method="bounded").x
print("temperature:", round(T, 3), "(<1 sharpens, >1 softens)")

# 3) refit on full train with the chosen C, apply T, evaluate on val
lr_t = make_pipeline(TfidfVectorizer(ngram_range=(1,2), sublinear_tf=True),
                     LogisticRegression(C=best["tfidf"], max_iter=3000)).fit(X_tr_t, y_tr)
lr_e = LogisticRegression(C=best["emb"], max_iter=3000).fit(E_tr, y_tr)
assert (lr_t.classes_ == lr_e.classes_).all() and (lr_t.classes_ == classes).all()

proba_val = sharpen((lr_t.predict_proba(X_va_t) + lr_e.predict_proba(E_va)) / 2, T)

res = pd.DataFrame([decide(p, classes, cfg) for p in proba_val])
res["correct"] = res.predicted.values == val.category.values
print(res.groupby("action").agg(n=("correct","size"), acc=("correct","mean")))
print(res.groupby("correct").confidence.describe())

In [ ]:
hw = pd.read_csv(DATA / "test_handwritten.csv")
print(hw[hw.split == "dev"][["id", "subject", "body"]].to_string())

In [ ]:
hw = pd.read_csv(DATA / "test_handwritten.csv", dtype={"subject": str, "body": str})
print(hw[hw.split == "dev"][["id"]].to_string())

In [ ]:
from sklearn.preprocessing import normalize

def max_sim(E_q, E_ref):
    return (normalize(E_q) @ normalize(E_ref).T).max(1)

val_sim = max_sim(E_va, E_tr)
print("val sim percentiles (1,5,10,25,50):", np.percentile(val_sim, [1, 5, 10, 25, 50]).round(3))

# Do low-similarity val emails tend to be the wrong ones?
res["sim"] = val_sim
print(res.groupby("correct").sim.describe().round(3))

In [ ]:
def topk_sim(E_q, E_ref, k=3):
    S = normalize(E_q) @ normalize(E_ref).T
    return np.sort(S, axis=1)[:, -k:].mean(1)

res["sim_top3"] = topk_sim(E_va, E_tr)
print("top3 percentiles (1,5,10,25,50):",
      np.percentile(res.sim_top3, [1, 5, 10, 25, 50]).round(3))
print(res.groupby("correct").sim_top3.describe().round(3))

In [ ]:
hw = pd.read_csv(DATA / "test_handwritten.csv", dtype={"subject": str, "body": str})
mask = hw["id"].str.endswith(("-messy-1", "-novel-1"))
hw.loc[mask, "split"] = "dev"
hw.to_csv(DATA / "test_handwritten.csv", index=False, encoding="utf-8")
print(hw.groupby(["split", "category"]).size().unstack(fill_value=0))

In [ ]:
# Cell removed from the public copy: it contained hand-written / personal email text.
# The hand-written test set is not published (see README).

In [ ]:
# Cell removed from the public copy: it contained hand-written / personal email text.
# The hand-written test set is not published (see README).

In [ ]:
hw = pd.read_csv(DATA / "test_handwritten.csv", dtype=str, keep_default_na=False)
hw["filled"] = hw["body"].str.len() > 0
print(hw.groupby(["split", "category"])["filled"].sum().unstack(fill_value=0))

In [ ]:
# Cell removed from the public copy: it contained hand-written / personal email text.
# The hand-written test set is not published (see README).

In [ ]:
from sklearn.preprocessing import normalize

hw = pd.read_csv(DATA / "test_handwritten.csv", dtype=str, keep_default_na=False, encoding="utf-8")
dev = hw[(hw.split == "dev") & (hw.body.str.len() > 0)].copy()
print("dev rows:", len(dev), "(expect 28)")

X_d_t = preprocess_df(dev, mode="tfidf")
X_d_e = preprocess_df(dev, mode="embedding")
E_d = st.encode(list(X_d_e))
P_d = sharpen((lr_t.predict_proba(X_d_t) + lr_e.predict_proba(E_d)) / 2, T)

dev["conf"] = P_d.max(1)
dev["pred"] = classes[P_d.argmax(1)]
dev["sim"] = (normalize(E_d) @ normalize(E_tr).T).max(1)

pd.set_option("display.width", 200)
print(dev[["id", "category", "pred", "conf", "sim"]].round(3).to_string(index=False))

In [ ]:
hw = pd.read_csv(DATA / "test_handwritten.csv", dtype=str, keep_default_na=False, encoding="utf-8")
hw["filled"] = hw["body"].str.len() > 0
print(hw.groupby(["split", "category"])["filled"].sum().unstack(fill_value=0))
print(hw.loc[hw.id == "comp-messy-1", ["split", "body"]].values)

In [ ]:
# Cell removed from the public copy: it contained hand-written / personal email text.
# The hand-written test set is not published (see README).

In [ ]:
from sklearn.preprocessing import normalize

hw = pd.read_csv(DATA / "test_handwritten.csv", dtype=str, keep_default_na=False, encoding="utf-8")
dev = hw[(hw.split == "dev") & (hw.body.str.len() > 0)].copy()
print("dev rows:", len(dev), "(expect 28)")

X_d_t = preprocess_df(dev, mode="tfidf")
X_d_e = preprocess_df(dev, mode="embedding")
E_d = st.encode(list(X_d_e))
P_d = sharpen((lr_t.predict_proba(X_d_t) + lr_e.predict_proba(E_d)) / 2, T)

dev["conf"] = P_d.max(1)
dev["pred"] = classes[P_d.argmax(1)]
dev["sim"] = (normalize(E_d) @ normalize(E_tr).T).max(1)

pd.set_option("display.width", 200)
print(dev[["id", "category", "pred", "conf", "sim"]].round(3).to_string(index=False))

In [ ]:
import importlib
from src import decision
importlib.reload(decision)
from src.decision import load_config, decide
cfg = load_config(ROOT / "config" / "pipeline.yaml")
print(cfg["thresholds"])

In [ ]:
out = pd.DataFrame([decide(P_d[i], classes, cfg, sim=dev.sim.iloc[i]) for i in range(len(dev))])
out["id"] = dev.id.values
out["true"] = dev.category.values
out["is_unk"] = out.true == "Unknown"
out["to_review"] = out.action == "manual_review"

print("Unknown caught:", out[out.is_unk].to_review.sum(), "/", out.is_unk.sum())
print("In-category wrongly sent to manual review:", out[~out.is_unk].to_review.sum(), "/", (~out.is_unk).sum())
print(out[["id", "true", "category", "action", "reason", "confidence"]].round(3).to_string(index=False))

In [ ]:
from sklearn.metrics import classification_report, f1_score, accuracy_score

pri_tr, pri_va = train["priority"], val["priority"]

# baseline: the most common priority within each (true) category
mode = train.groupby("category")["priority"].agg(lambda s: s.mode()[0])
base = val["category"].map(mode)
print("category-only baseline: acc %.3f | macro F1 %.3f" %
      (accuracy_score(pri_va, base), f1_score(pri_va, base, average="macro")))

pri_model = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
    LogisticRegression(max_iter=3000, C=10, class_weight="balanced"))
pri_model.fit(X_tr_t, pri_tr)
pri_pred = pri_model.predict(X_va_t)
print(classification_report(pri_va, pri_pred, digits=3))

In [ ]:
mask = (pri_va.values == "High") & (pri_pred != "High")
miss = val[mask].copy()
miss["pred_priority"] = pri_pred[mask]
pd.set_option("display.max_colwidth", 230); pd.set_option("display.width", 250)
print(miss[["category", "pred_priority", "subject", "body"]].to_string())

tf = pri_model.steps[0][1]; clf = pri_model.steps[-1][1]
coef = clf.coef_[list(clf.classes_).index("High")]
names = np.array(tf.get_feature_names_out())
print("\nTop terms for High:", list(names[np.argsort(coef)[-25:][::-1]]))

In [ ]:
import re
from sklearn.model_selection import StratifiedKFold, cross_val_predict

def raw_text(df):
    return (df["subject"].fillna("") + " " + df["body"].fillna("")).str.lower()

RULES = {
 "urgent":     r"\burgent(ly)?\b|\bemergency\b",
 "asap":       r"\basap\b|as soon as possible|\bimmediate(ly)?\b|right away|right now|at once",
 "deadline":   r"\b(today|tonight|tomorrow)\b|\bby \d{1,2}(:\d\d)? ?(am|pm)\b|\bby (end of (the )?day|eod)\b|within (24|48) hours",
 "escalation": r"escalat|legal action|consumer (forum|court)|regulator|social media|public review|lawyer|chargeback",
 "security":   r"\bhack|unauthori[sz]ed|\bsomeone (changed|logged|accessed|else)|isn't mine|\bnot me\b|suspicious|compromis|\bscam|fraud|stolen",
 "money_lost": r"(charged|debited|billed)( me)? (twice|two times|2x)|double (charged|debited|billing)|(deducted|debited|taken out|gone from)|money (was )?(deducted|debited|gone|taken)",
 "outage":     r"\bdown\b|outage|nobody (can|has been able)|none of our|whole team|all (our )?(users|staff|employees)|crash(es|ed|ing)?\b",
 "safety":     r"gas (smell|leak)|\bfire\b|smoke|injur|ambulance|stuck (in|between)|medication|elderly|\bunsafe\b",
}
NEG = r"\b(not|no|isn't|nothing)\b[^.]{0,15}\b(urgent|rush|hurry)\b|low priority|whenever (you|it)"

def rule_hits(df):
    t = raw_text(df)
    H = pd.DataFrame({k: t.str.contains(v, regex=True) for k, v in RULES.items()})
    H.loc[t.str.contains(NEG, regex=True), ["urgent", "asap", "deadline"]] = False
    return H

# ---- 1) Evaluate each rule on TRAIN, using out-of-fold ML predictions ----
skf = StratifiedKFold(5, shuffle=True, random_state=42)
oof_pri = cross_val_predict(pri_model, X_tr_t, pri_tr, cv=skf)
H_tr = rule_hits(train)
is_high = (pri_tr.values == "High")
ml_not_high = (oof_pri != "High")

rows = []
for k in H_tr:
    m = H_tr[k].values
    mm = m & ml_not_high                      # rule fires where ML says non-High
    gain, loss = (mm & is_high).sum(), (mm & ~is_high).sum()
    rows.append(dict(rule=k, fires=m.sum(), high_rate=round(is_high[m].mean(), 2) if m.sum() else np.nan,
                     override_fires=mm.sum(), gain=gain, loss=loss,
                     override_precision=round(gain / mm.sum(), 2) if mm.sum() else np.nan))
print("High base rate in train:", is_high.mean().round(3))
print(pd.DataFrame(rows).to_string(index=False))
print("ML-only High recall (OOF, train):", ((oof_pri == "High") & is_high).sum() / is_high.sum())

In [ ]:
def combo(keep):
    if keep:
        fired = H_tr[keep].any(axis=1).values
        hy = np.where(fired, "High", oof_pri)
    else:
        hy = oof_pri
    ch = hy != oof_pri
    tp = ((hy == "High") & is_high).sum()
    return dict(rules="+".join(keep) or "ML only", overrides=ch.sum(),
                gain=(ch & is_high).sum(), loss=(ch & ~is_high).sum(),
                high_recall=round(tp / is_high.sum(), 3),
                high_prec=round(tp / (hy == "High").sum(), 3),
                macro_f1=round(f1_score(pri_tr.values, hy, average="macro"), 3))

cands = [[], ["asap", "escalation", "money_lost"],
         ["asap", "escalation", "money_lost", "urgent"],
         ["asap", "escalation", "money_lost", "urgent", "security"]]
print(pd.DataFrame([combo(k) for k in cands]).to_string(index=False))

In [ ]:
KEEP = ["asap", "escalation", "money_lost", "urgent", "security"]

def hybrid_priority(df, ml_pred):
    H = rule_hits(df)[KEEP]
    fired = H.any(axis=1).values
    why = H.apply(lambda r: ", ".join(r.index[r]), axis=1).values
    return np.where(fired, "High", ml_pred), why

ml_va = pri_model.predict(X_va_t)
hy_va, why = hybrid_priority(val, ml_va)
y = pri_va.values

for name, p in [("ML only", ml_va), ("Hybrid", hy_va)]:
    tp = ((p == "High") & (y == "High")).sum()
    print(f"{name:8s} acc {accuracy_score(y, p):.3f} | macro F1 {f1_score(y, p, average='macro'):.3f} "
          f"| High recall {tp/(y=='High').sum():.2f} precision {tp/(p=='High').sum():.2f}")

ch = hy_va != ml_va
print(f"overrides: {ch.sum()} | became correct: {((hy_va == y) & ch).sum()} | became wrong: {((hy_va != y) & ch).sum()}")

In [ ]:
import importlib
from src import priority
importlib.reload(priority)

hy2, why2 = priority.hybrid_priority(val, ml_va)
print("identical to before:", (hy2 == hy_va).all(),
      "| acc", round(accuracy_score(y, hy2), 3),
      "| macro F1", round(f1_score(y, hy2, average="macro"), 3))

In [ ]:
def top_terms(text_t, pipe, pred_class, k=5):
    vec, clf = pipe.steps[0][1], pipe.steps[-1][1]
    x = vec.transform([text_t]).toarray()[0]
    contrib = x * clf.coef_[list(clf.classes_).index(pred_class)]   # tfidf weight x class coefficient
    names = vec.get_feature_names_out()
    order = np.argsort(contrib)[::-1][:k]
    return [(names[i], round(float(contrib[i]), 2)) for i in order if contrib[i] > 0]

for i in range(5):
    pred = res.predicted.iloc[i]
    print(f"[{i}] true: {val.category.iloc[i]} | pred: {pred} | conf: {res.confidence.iloc[i]:.2f}")
    print("    ", val.subject.iloc[i])
    print("    ", top_terms(X_va_t.iloc[i] if hasattr(X_va_t, "iloc") else X_va_t[i], lr_t, pred))

In [ ]:
from sklearn.preprocessing import normalize
En_tr = normalize(E_tr)

def nearest_examples(e, k=3):
    s = En_tr @ normalize(e.reshape(1, -1))[0]
    idx = np.argsort(s)[::-1][:k]
    return [(train.category.iloc[j], round(float(s[j]), 2), train.subject.iloc[j][:60]) for j in idx]

for i in range(5):
    print(f"[{i}] pred: {res.predicted.iloc[i]} | {val.subject.iloc[i]}")
    for c, s, subj in nearest_examples(E_va[i]):
        print(f"     {s:.2f} | {c} | {subj}")

In [ ]:
rv = pd.DataFrame([decide(proba_val[i], classes, cfg, sim=val_sim[i]) for i in range(len(val))])
rv["correct"] = rv.predicted.values == val.category.values

unk = rv.category == "Unknown"
print(f"val emails sent to Unknown by the gate: {unk.sum()} / {len(rv)} ({unk.mean():.1%})")
print("of those, the model's own prediction was right:", rv[unk].correct.sum())
print(rv.groupby("action").size())

In [ ]:
rows = []
for t in [0.36, 0.38, 0.40, 0.42, 0.44, 0.46, 0.48]:
    val_flag = (val_sim < t)
    d_unk = dev[dev.category == "Unknown"]
    d_in = dev[dev.category != "Unknown"]
    rows.append(dict(threshold=t,
                     val_false_unknown=f"{val_flag.sum()} ({val_flag.mean():.1%})",
                     dev_unknown_caught=f"{(d_unk.sim < t).sum()} / {len(d_unk)}",
                     dev_in_category_flagged=f"{(d_in.sim < t).sum()} / {len(d_in)}"))
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
cfg = load_config(ROOT / "config" / "pipeline.yaml")
print(cfg["thresholds"])

rv = pd.DataFrame([decide(proba_val[i], classes, cfg, sim=val_sim[i]) for i in range(len(val))])
rv["correct"] = rv.predicted.values == val.category.values
print(rv.groupby("action").agg(n=("correct", "size"), acc=("correct", "mean")).round(3))

In [ ]:
p = ROOT / "config" / "pipeline.yaml"
print(p)
print(p.read_text())

In [ ]:
cfg = load_config(ROOT / "config" / "pipeline.yaml")
print(cfg["thresholds"])

In [ ]:
rv = pd.DataFrame([decide(proba_val[i], classes, cfg, sim=val_sim[i]) for i in range(len(val))])
rv["correct"] = rv.predicted.values == val.category.values
print(rv.groupby("action").agg(n=("correct", "size"), acc=("correct", "mean")).round(3))

In [ ]:
import joblib
M = ROOT / "models"
M.mkdir(exist_ok=True)

joblib.dump(lr_t, M / "tfidf_lr.joblib")
joblib.dump(lr_e, M / "minilm_lr.joblib")
joblib.dump(pri_model, M / "priority_model.joblib")
joblib.dump({"T": float(T), "classes": list(classes)}, M / "meta.joblib")
np.save(M / "E_train.npy", E_tr)

print(sorted(p.name for p in M.iterdir()))
print("E_train rows match train.csv:", E_tr.shape[0] == len(train), "| T =", round(float(T), 3))

In [ ]:
import importlib
from src import predict as pm
importlib.reload(pm)

tests = [("Change username", "How do I change my username? Thanks"),
         ("URGENT: charged twice", "I was charged twice for order 5521, please refund asap"),
         ("weather", "what's the weather going to be like in Delhi tomorrow?")]

for sub, body in tests:
    r = pm.predict(sub, body)
    print(r["category"], "|", r["department"], "|", r["action"], "|", round(r["confidence"], 2),
          "|", r["priority"], f"({r['priority_reason']})")
    print("   terms:", r["top_terms"])
    print("   neighbours:", r["neighbours"])

In [ ]:
import importlib
from src import notices
importlib.reload(notices)

# 1) False positives: real emails (train + val) that would be filtered
trval = pd.concat([train, val], ignore_index=True)
flags = [notices.detect_notice(s, b) for s, b in zip(trval.subject, trval.body)]
trval["filtered"] = [f[0] for f in flags]
print("real emails that would be filtered:", trval.filtered.sum(), "/", len(trval))
print(trval[trval.filtered][["category", "subject"]].to_string())

# 2) Your dev hand-written emails
hw = pd.read_csv(DATA / "test_handwritten.csv", dtype=str, keep_default_na=False)
dev = hw[(hw.split == "dev") & (hw.body.str.len() > 0)]
for _, r in dev.iterrows():
    ok, h = notices.detect_notice(r.subject, r.body)
    print(f"{r.id:18s} {r.category:18s} filtered={ok!s:5s} {h}")

In [ ]:
importlib.reload(notices)

# 1) False positives on real emails (must stay 0)
flags = [notices.detect_notice(s, b)[0] for s, b in zip(trval.subject, trval.body)]
print("real emails that would be filtered:", sum(flags), "/", len(trval))

# 2) The Google email
g_sub = "You shared some Google Account data with Claude"
g_body = ("You're receiving this email because you used Sign in with Google to sign in to Claude. "
          "This email summarizes the info you shared. There's nothing you need to do right now. "
          "If you want to stop receiving these emails, you can unsubscribe. © 2026 Google LLC")
print(notices.detect_notice(g_sub, g_body))

# 3) Dev emails still behave
for _, r in dev.iterrows():
    ok, h = notices.detect_notice(r.subject, r.body)
    if ok or r.category == "Unknown":
        print(f"{r.id:18s} filtered={ok!s:5s} {h}")

In [ ]:
val[["subject", "body"]].head(10).to_csv(DATA / "batch_demo.csv", index=False, encoding="utf-8")

In [ ]:
hw = pd.read_csv(DATA / "test_handwritten.csv", dtype=str, keep_default_na=False)
todo = hw[(hw.split == "test") & (hw.body.str.len() == 0)]
print(len(todo), "rows to write")
print(todo.groupby("category").id.apply(list).to_string())

In [ ]:
import importlib, time
from src import predict as pm
importlib.reload(pm)
from sklearn.metrics import classification_report, accuracy_score, f1_score

test = pd.read_csv(DATA / "test.csv")
hw = pd.read_csv(DATA / "test_handwritten.csv", dtype=str, keep_default_na=False, encoding="utf-8")
hwt = hw[(hw.split == "test") & (hw.body.str.len() > 0)].copy()
print(len(test), "held-out emails |", len(hwt), "hand-written test emails")

def run(df):
    t0 = time.time()
    out = [pm.predict(s, b) for s, b in zip(df.subject.fillna(""), df.body.fillna(""))]
    ms = (time.time() - t0) / len(df) * 1000
    r = pd.DataFrame(out, index=df.index)
    d = df.copy()
    d["pred"], d["action"], d["conf"], d["pred_priority"] = r.category, r.action, r.confidence, r.priority
    return d, ms

t, ms_t = run(test)
h, ms_h = run(hwt)
os.makedirs(ROOT / "reports", exist_ok=True)
t.to_csv(ROOT / "reports" / "final_test_predictions.csv", index=False)
h.to_csv(ROOT / "reports" / "final_handwritten_predictions.csv", index=False)
print(f"speed: {ms_t:.0f} ms per email (full pipeline, one at a time)")

print("\n=== HELD-OUT TEST (synthetic + Kaggle) ===")
print(classification_report(t.category, t.pred, digits=3))
print(t.groupby("action").apply(lambda g: pd.Series({"n": len(g), "acc": round((g.pred == g.category).mean(), 3)})))
print("by source:", t.groupby("source").apply(lambda g: round((g.pred == g.category).mean(), 3)).to_dict())
print("priority: acc %.3f | macro F1 %.3f" % (accuracy_score(t.priority, t.pred_priority),
      f1_score(t.priority, t.pred_priority, average="macro")))

print("\n=== HAND-WRITTEN TEST ===")
inc = h[h.category != "Unknown"].copy()
inc["strict"] = inc.pred == inc.category
inc["lenient"] = inc.strict | (inc.pred == inc.alt_label)
print("in-category: strict %.3f | lenient %.3f (n=%d)" % (inc.strict.mean(), inc.lenient.mean(), len(inc)))
print(inc.groupby("type")[["strict", "lenient"]].mean().round(3))
unk = h[h.category == "Unknown"]
print("Unknown caught (pred = Unknown): %d / %d" % ((unk.pred == "Unknown").sum(), len(unk)))
print("in-category emails wrongly sent to Unknown: %d / %d" % ((inc.pred == "Unknown").sum(), len(inc)))
print(classification_report(h.category, h.pred, digits=3, zero_division=0))
print(pd.crosstab(h.category, h.pred))
print("priority (in-category): acc %.3f" % accuracy_score(inc.priority, inc.pred_priority))

In [ ]:
from sklearn.metrics import accuracy_score

# A) Does predict() reproduce the notebook's validation results?
v = pd.DataFrame([pm.predict(s, b) for s, b in zip(val.subject.fillna(""), val.body.fillna(""))], index=val.index)
print("val category acc via predict():", round((v.category == val.category).mean(), 3))
print("val priority acc via predict():", round(accuracy_score(val.priority, v.priority), 3), "(notebook hybrid: 0.775)")

# B) Every hand-written error, with how it was handled
print(h[h.category != h.pred][["id", "category", "pred", "action", "conf", "subject"]].round(2).to_string(index=False))
print(h.groupby("action").apply(lambda g: pd.Series({"n": len(g), "strict_acc": round((g.pred == g.category).mean(), 3)})))

# C) Held-out priority: by action, and confusion
t["pri_ok"] = t.priority == t.pred_priority
print(t.groupby("action").pri_ok.agg(["mean", "size"]).round(3))
print(pd.crosstab(t.priority, t.pred_priority))

In [ ]:
import re
PH = re.compile(r"Tech Online Store\s*", re.I)
def strip_ph(df):
    d = df.copy()
    for c in ["subject", "body"]:
        d[c] = d[c].str.replace(PH, "", regex=True)
    return d

def fit_eval(tr, va):
    pipe = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
                         LogisticRegression(max_iter=3000, C=10))
    pipe.fit(preprocess_df(tr), tr["category"])
    p = pipe.predict(preprocess_df(va))
    return round(accuracy_score(va["category"], p), 3), round(f1_score(va["category"], p, average="macro"), 3)

print("train as is   | val as is      :", fit_eval(train, val))
print("train as is   | val stripped    :", fit_eval(train, strip_ph(val)))
print("train stripped | val stripped   :", fit_eval(strip_ph(train), strip_ph(val)))

In [ ]:
hw = pd.read_csv(DATA / "test_handwritten.csv", dtype=str, keep_default_na=False, encoding="utf-8")
hw["filled"] = hw.body.str.len() > 0
print(hw.groupby(["split", "category"]).filled.sum().unstack(fill_value=0))

In [ ]:
import re
PH = re.compile(r"Tech Online Store\s*", re.I)
def strip_ph(df):
    d = df.copy()
    for c in ["subject", "body"]:
        d[c] = d[c].str.replace(PH, "", regex=True)
    return d

def fit_eval(tr, va):
    pipe = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
                         LogisticRegression(max_iter=3000, C=10))
    pipe.fit(preprocess_df(tr), tr["category"])
    p = pipe.predict(preprocess_df(va))
    return round(accuracy_score(va["category"], p), 3), round(f1_score(va["category"], p, average="macro"), 3)

print("train as is   | val as is      :", fit_eval(train, val))
print("train as is   | val stripped    :", fit_eval(train, strip_ph(val)))
print("train stripped | val stripped   :", fit_eval(strip_ph(train), strip_ph(val)))

In [ ]:
for name, d in [("train", train), ("val", val)]:
    print(name, (d.subject + " " + d.body).str.contains("Tech Online Store", case=False).sum(), "rows contain the phrase")

In [ ]:
test = pd.read_csv("../data/test.csv", keep_default_na=False)
print("test", (test.subject + " " + test.body).str.contains("Tech Online Store", case=False).sum(), "rows contain the phrase")